In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
train_data = pd.read_csv("../data/processed/train.csv")
test_data = pd.read_csv("../data/processed/test.csv")

In [3]:


# Ngưỡng xác định phim tích cực
POSITIVE_THRESHOLD = 4

# Lấy các tương tác tích cực từ tập Train
positive_data = train_data[
    train_data["rating"] >= POSITIVE_THRESHOLD
].copy()

# Ánh xạ user và movie sang chỉ số liên tục
user_ids = np.sort(train_data["user_id"].unique())
movie_ids = np.sort(train_data["movie_id"].unique())

user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
movie_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)}

positive_data["user_idx"] = positive_data["user_id"].map(user_to_idx)
positive_data["movie_idx"] = positive_data["movie_id"].map(movie_to_idx)

positive_users = positive_data["user_idx"].to_numpy(dtype=np.int32)
positive_movies = positive_data["movie_idx"].to_numpy(dtype=np.int32)

n_users = len(user_ids)
n_movies = len(movie_ids)

print("Số user:", n_users)
print("Số movie:", n_movies)
print("Tổng tương tác trong Train:", len(train_data))
print("Số tương tác tích cực:", len(positive_data))
print("Tỷ lệ tương tác tích cực:", f"{len(positive_data) / len(train_data):.2%}")
print("Số user có tương tác tích cực:", positive_data["user_id"].nunique())
print("Số movie có tương tác tích cực:", positive_data["movie_id"].nunique())

Số user: 6040
Số movie: 3683
Tổng tương tác trong Train: 800167
Số tương tác tích cực: 460119
Tỷ lệ tương tác tích cực: 57.50%
Số user có tương tác tích cực: 6037
Số movie có tương tác tích cực: 3499


In [4]:
# Tập phim đã tương tác của mỗi user, bao gồm mọi mức rating
seen_movies_by_user = (
    train_data.groupby("user_id")["movie_id"]
    .apply(lambda x: set(x))
    .to_dict()
)

# Chuyển sang chỉ số movie liên tục để dùng trong huấn luyện
seen_movie_indices = {
    user_to_idx[user_id]: {
        movie_to_idx[movie_id]
        for movie_id in movies
        if movie_id in movie_to_idx
    }
    for user_id, movies in seen_movies_by_user.items()
}

# Kiểm tra dữ liệu
sample_user_id = int(user_ids[0])
sample_user_idx = user_to_idx[sample_user_id]

print("Số user có lịch sử tương tác:", len(seen_movie_indices))
print("User ID mẫu:", sample_user_id)
print("Số phim user đã tương tác trong Train:",
      len(seen_movie_indices[sample_user_idx]))
print("Số phim ứng viên chưa tương tác:",
      n_movies - len(seen_movie_indices[sample_user_idx]))

Số user có lịch sử tương tác: 6040
User ID mẫu: 1
Số phim user đã tương tác trong Train: 41
Số phim ứng viên chưa tương tác: 3642


In [5]:
def sample_negative_item(user_idx, seen_movie_indices, n_movies, rng):
    seen = seen_movie_indices[user_idx]

    while True:
        negative_idx = int(rng.integers(0, n_movies))
        if negative_idx not in seen:
            return negative_idx

In [6]:
rng = np.random.default_rng(42)

sample_positive_user = int(positive_users[0])
sample_negative = sample_negative_item(
    sample_positive_user,
    seen_movie_indices,
    n_movies,
    rng
)

print("User index:", sample_positive_user)
print("Negative movie index:", sample_negative)
print(
    "Phim âm tính chưa xuất hiện trong Train:",
    sample_negative not in seen_movie_indices[sample_positive_user]
)

User index: 4086
Negative movie index: 328
Phim âm tính chưa xuất hiện trong Train: True


In [7]:
N_FACTORS = 50
LEARNING_RATE = 0.01
REGULARIZATION = 0.01
N_EPOCHS = 15
RANDOM_STATE = 42

print("Số latent factors:", N_FACTORS)
print("Learning rate:", LEARNING_RATE)
print("Regularization:", REGULARIZATION)
print("Số epochs:", N_EPOCHS)
print("Số positive interactions:", len(positive_users))

Số latent factors: 50
Learning rate: 0.01
Regularization: 0.01
Số epochs: 15
Số positive interactions: 460119


In [8]:
def train_bpr_mf(
    positive_users,
    positive_movies,
    seen_movie_indices,
    n_users,
    n_movies,
    n_factors=50,
    learning_rate=0.01,
    reg=0.01,
    n_epochs=15,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    # Khởi tạo vector tiềm ẩn
    user_factors = rng.normal(
        0, 0.05, (n_users, n_factors)
    ).astype(np.float32)

    movie_factors = rng.normal(
        0, 0.05, (n_movies, n_factors)
    ).astype(np.float32)

    n_samples = len(positive_users)
    history = []

    for epoch in range(n_epochs):
        order = rng.permutation(n_samples)
        total_loss = 0.0

        for idx in order:
            u = int(positive_users[idx])
            i = int(positive_movies[idx])

            # Lấy một phim âm tính chưa được user tương tác
            j = sample_negative_item(
                u, seen_movie_indices, n_movies, rng
            )

            # Sao chép vector trước khi cập nhật
            p_u = user_factors[u].copy()
            q_i = movie_factors[i].copy()
            q_j = movie_factors[j].copy()

            # Chênh lệch điểm xếp hạng giữa positive và negative
            x_uij = np.dot(p_u, q_i - q_j)

            # Gradient của -log(sigmoid(x_uij))
            # Công thức ổn định số học
            coeff = 1.0 / (1.0 + np.exp(np.clip(x_uij, -20, 20)))

            # Cập nhật các vector tiềm ẩn
            user_factors[u] += learning_rate * (
                coeff * (q_i - q_j) - reg * p_u
            )

            movie_factors[i] += learning_rate * (
                coeff * p_u - reg * q_i
            )

            movie_factors[j] += learning_rate * (
                -coeff * p_u - reg * q_j
            )

            # BPR loss kèm regularization
            loss = np.logaddexp(0.0, -x_uij)
            loss += reg * (
                np.sum(p_u ** 2)
                + np.sum(q_i ** 2)
                + np.sum(q_j ** 2)
            )
            total_loss += loss

        avg_loss = total_loss / n_samples
        history.append(avg_loss)

        print(
            f"Epoch {epoch + 1:02d}/{n_epochs} | "
            f"BPR Loss: {avg_loss:.4f}"
        )

    return {
        "user_factors": user_factors,
        "movie_factors": movie_factors,
        "history": history
    }

In [9]:
bpr_model = train_bpr_mf(
    positive_users=positive_users,
    positive_movies=positive_movies,
    seen_movie_indices=seen_movie_indices,
    n_users=n_users,
    n_movies=n_movies,
    n_factors=N_FACTORS,
    learning_rate=LEARNING_RATE,
    reg=REGULARIZATION,
    n_epochs=N_EPOCHS,
    random_state=RANDOM_STATE
)

Epoch 01/15 | BPR Loss: 0.6967
Epoch 02/15 | BPR Loss: 0.6954
Epoch 03/15 | BPR Loss: 0.6891
Epoch 04/15 | BPR Loss: 0.6462
Epoch 05/15 | BPR Loss: 0.5325
Epoch 06/15 | BPR Loss: 0.4426
Epoch 07/15 | BPR Loss: 0.4023
Epoch 08/15 | BPR Loss: 0.3805
Epoch 09/15 | BPR Loss: 0.3696
Epoch 10/15 | BPR Loss: 0.3605
Epoch 11/15 | BPR Loss: 0.3554
Epoch 12/15 | BPR Loss: 0.3490
Epoch 13/15 | BPR Loss: 0.3448
Epoch 14/15 | BPR Loss: 0.3403
Epoch 15/15 | BPR Loss: 0.3362


In [10]:
def recommend_bpr_top_n(user_id, model, n=10):
    if user_id not in user_to_idx:
        return pd.DataFrame(columns=["movie_id", "score"])

    u = user_to_idx[user_id]

    # Tính điểm xếp hạng cho toàn bộ phim
    scores = (
        model["movie_factors"]
        @ model["user_factors"][u]
    )

    # Loại phim đã xuất hiện trong Train
    seen_movies = seen_movies_by_user.get(user_id, set())

    candidate_mask = ~np.isin(movie_ids, list(seen_movies))
    candidate_movies = movie_ids[candidate_mask]
    candidate_scores = scores[candidate_mask]

    top_indices = np.argsort(candidate_scores)[-n:][::-1]

    return pd.DataFrame({
        "movie_id": candidate_movies[top_indices],
        "score": candidate_scores[top_indices]
    }).reset_index(drop=True)

In [11]:
sample_user = int(train_data["user_id"].iloc[0])

bpr_recommendations = recommend_bpr_top_n(
    user_id=sample_user,
    model=bpr_model,
    n=10
)

print("User ID:", sample_user)
display(bpr_recommendations)

User ID: 2507


,movie_id,score
0,593,4.106958
1,2396,3.984998
2,1210,3.798397
3,2028,3.785219
4,1198,3.722592
5,1221,3.722355
6,919,3.705059
7,750,3.663894
8,1259,3.648298
9,2791,3.464872


In [12]:
def evaluate_bpr_top_n(
    model,
    eval_users,
    relevant_items_by_user,
    n=10
):
    precisions, recalls, f1_scores, ndcgs = [], [], [], []

    # Lịch sử phim đã tương tác trong Train
    seen_by_user = (
        train_data.groupby("user_id")["movie_id"]
        .apply(set)
        .to_dict()
    )

    for user_id in eval_users:
        u = user_to_idx[user_id]

        # BPR score = tích vô hướng user-factor và movie-factor
        scores = (
            model["movie_factors"]
            @ model["user_factors"][u]
        )

        seen_movies = seen_by_user.get(user_id, set())
        candidate_mask = ~np.isin(movie_ids, list(seen_movies))

        candidate_movies = movie_ids[candidate_mask]
        candidate_scores = scores[candidate_mask]

        top_indices = np.argsort(candidate_scores)[-n:][::-1]
        top_movies = candidate_movies[top_indices]

        relevant_items = relevant_items_by_user[user_id]
        hits = [
            int(movie_id in relevant_items)
            for movie_id in top_movies
        ]

        n_hits = sum(hits)
        precision = n_hits / n
        recall = n_hits / len(relevant_items)

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        dcg = sum(
            hit / np.log2(rank + 2)
            for rank, hit in enumerate(hits)
        )

        ideal_hits = min(n, len(relevant_items))
        idcg = sum(
            1 / np.log2(rank + 2)
            for rank in range(ideal_hits)
        )

        ndcg = dcg / idcg if idcg > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        ndcgs.append(ndcg)

    return {
        f"Precision@{n}": float(np.mean(precisions)),
        f"Recall@{n}": float(np.mean(recalls)),
        f"F1@{n}": float(np.mean(f1_scores)),
        f"NDCG@{n}": float(np.mean(ndcgs))
    }

In [15]:
# Lọc những user và movie đã xuất hiện trong Train
test_eval = test_data[
    test_data["user_id"].isin(user_to_idx)
    & test_data["movie_id"].isin(movie_to_idx)
].copy()

print("Số dòng trong Test ban đầu:", len(test_data))
print("Số dòng dùng để đánh giá:", len(test_eval))
print("Số user được giữ lại:", test_eval["user_id"].nunique())
print("Số movie được giữ lại:", test_eval["movie_id"].nunique())

Số dòng trong Test ban đầu: 200042
Số dòng dùng để đánh giá: 200016
Số user được giữ lại: 6038
Số movie được giữ lại: 3421


In [16]:
# Xác định các phim liên quan trong tập Test
RELEVANCE_THRESHOLD = 4

relevant_test = test_eval[
    test_eval["rating"] >= RELEVANCE_THRESHOLD
].copy()

relevant_items_by_user = (
    relevant_test.groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)

# Chỉ đánh giá user có ít nhất một phim liên quan trong Test
eval_users = sorted(relevant_items_by_user.keys())

print("Số user được đánh giá:", len(eval_users))
print("Số user có phim liên quan trong Test:", len(relevant_items_by_user))
print("Số tương tác tích cực trong Test:", len(relevant_test))

Số user được đánh giá: 5994
Số user có phim liên quan trong Test: 5994
Số tương tác tích cực trong Test: 115155


In [17]:
bpr_results = evaluate_bpr_top_n(
    model=bpr_model,
    eval_users=eval_users,
    relevant_items_by_user=relevant_items_by_user,
    n=10
)

print("Kết quả BPR-MF:")
for metric, value in bpr_results.items():
    print(f"{metric}: {value:.4f}")

Kết quả BPR-MF:
Precision@10: 0.1666
Recall@10: 0.0968
F1@10: 0.0992
NDCG@10: 0.1928
